In [ ]:
# ============================================================================
# CONFIGURATION - EDIT THESE VALUES
# ============================================================================

REPO_URL = "https://github.com/YOUR_USERNAME/MetaHackUI.git"  # 👈 UPDATE THIS
RUN_SFT_TRAINING = True         # Set to False to skip SFT
RUN_RL_TRAINING = True          # Set to False to skip RL

# Output directories
OUTPUT_DIR = "/content/drive/My Drive/MetaHackUI_results"
RL_OUTPUT_DIR = "/content/drive/My Drive/MetaHackUI_RL_results"

print("=" * 70)
print("🔥 MetaHackUI Complete Training Pipeline")
print("=" * 70)
print(f"\nConfiguration:")
print(f"  SFT Training: {RUN_SFT_TRAINING}")
print(f"  RL Training: {RUN_RL_TRAINING}")
print(f"  Repository: {REPO_URL}")
print("=" * 70)

---

# ⚙️ CONFIGURATION
**UPDATE THESE VALUES:**
```
REPO_URL = "https://github.com/YOUR_USERNAME/MetaHackUI.git"  # 👈 UPDATE THIS
RUN_SFT_TRAINING = True         # Set to False to skip SFT
RUN_RL_TRAINING = True          # Set to False to skip RL
```

---

# 🔥 MetaHackUI - Complete Training on Google Colab
## SFT Fine-tuning + RL Policy Training in One Notebook

Run both training pipelines or choose SFT or RL training independently.

# ✅ Step 1: GPU Setup & Verification

In [ ]:
import torch
import subprocess

print("🔍 GPU Status:")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}GB")
    print(f"PyTorch: {torch.__version__}")
else:
    print("⚠️ WARNING: No GPU detected!")
    print("   Go to Runtime → Change Runtime Type → Select GPU (T4 or L4)")

# ✅ Step 2: Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

# Clone repository
if not os.path.exists("/content/MetaHackUI"):
    print(f"📦 Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, "/content/MetaHackUI"], check=True)
    os.chdir("/content/MetaHackUI")
else:
    os.chdir("/content/MetaHackUI")

print(f"✅ Working directory: {os.getcwd()}")
print(f"📂 Project structure:")
for folder in ['agents', 'openenv', 'rl', 'dataset', 'project']:
    exists = "✅" if os.path.exists(folder) else "❌"
    print(f"   {exists} {folder}/")

# ✅ Step 3: Install All Dependencies

In [ ]:
import sys

print("📦 Installing dependencies...")

packages = [
    # SFT Training
    "transformers",
    "datasets",
    "peft",
    "accelerate",
    "bitsandbytes",
    # RL Training
    "gymnasium>=0.29.0",
    "stable-baselines3>=2.0.0",
    # Core
    "torch>=2.0.0",
    "pydantic>=2.0.0",
    "python-dotenv",
    "aiohttp",
    "numpy",
    # Viz
    "matplotlib",
    "seaborn",
    "pandas",
    "tensorboard",
    "sentence-transformers",
    "faiss-cpu",
    "langchain",
    "rich"
]

for i, package in enumerate(packages, 1):
    pkg_name = package.split(">=")[0].split("<")[0]
    print(f"  [{i}/{len(packages)}] {pkg_name}...", end=" ", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print("✅")

print("\n✅ All dependencies installed!")

---

# 🟦 PART 1: SUPERVISED FINE-TUNING (SFT)
*Fine-tune Qwen2.5-3B-Instruct on cybersecurity data*

---

## ✅ SFT Step 1: Load & Prepare Data

In [ ]:
if RUN_SFT_TRAINING:
    print("\n" + "="*70)
    print("🟦 SFT TRAINING: Data Preparation")
    print("="*70)
    
    import json
    from datasets import load_dataset
    
    # Load dataset
    data_path = "/content/MetaHackUI/project/train.jsonl"
    if os.path.exists(data_path):
        dataset = load_dataset("json", data_files=data_path)["train"]
        print(f"✅ Loaded {len(dataset)} training examples")
        
        if len(dataset) > 0:
            print(f"\n📋 Sample keys: {list(dataset[0].keys())}")
    else:
        print(f"⚠️ {data_path} not found")
        dataset = None
else:
    print("⏭️ Skipping SFT training")

## ✅ SFT Step 2: Load Model & Setup LoRA

In [ ]:
if RUN_SFT_TRAINING and dataset:
    print("🤖 Loading model and tokenizer...")
    
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import LoraConfig, get_peft_model
    
    model_name = "Qwen/Qwen2.5-3B-Instruct"
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    print(f"✅ Tokenizer loaded")
    
    # Load model
    print("⏳ Loading Qwen2.5-3B (this takes ~1-2 min)...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    print(f"✅ Model loaded")
    print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
    
    # Memory optimization
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    
    # Configure LoRA
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    model = get_peft_model(model, lora_config)
    print(f"✅ LoRA applied")
    print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.2f}M")

## ✅ SFT Step 3: Format & Tokenize Dataset

In [ ]:
if RUN_SFT_TRAINING and dataset:
    print("\n📝 Formatting and tokenizing dataset...")
    
    def format_example(example):
        logs_str = json.dumps(example["logs"], indent=2)
        reqs_str = ", ".join(example["requirements"])
        code_str = example["code"]
        
        input_text = f"""You are a cybersecurity model.
Detect if an attack occurred and return structured JSON.

Logs:
{logs_str}

Requirements:
{reqs_str}

Code:
{code_str}
"""
        
        gt = example["known_truth"]
        output_text = json.dumps({
            "attack_type": gt["attack_type"],
            "flagged_logs": gt["flagged_logs"],
            "flagged_reqs": gt.get("flagged_requirements", gt.get("flagged_reqs", [])),
            "flagged_code": gt["flagged_code"]
        }, indent=2)
        
        full_text = input_text + "\n\n" + output_text
        return {"text": full_text}
    
    dataset = dataset.map(format_example, desc="Formatting")
    print(f"✅ Formatted {len(dataset)} examples")
    
    # Tokenize
    def tokenize(example):
        tokens = tokenizer(
            example["text"],
            truncation=True,
            padding="max_length",
            max_length=512
        )
        tokens["labels"] = tokens["input_ids"].copy()
        return tokens
    
    dataset = dataset.map(tokenize, batched=True, batch_size=32, desc="Tokenizing")
    dataset = dataset.remove_columns(["text"])
    print(f"✅ Tokenized {len(dataset)} examples")

## ✅ SFT Step 4: Train Model

In [ ]:
if RUN_SFT_TRAINING and dataset:
    from transformers import TrainingArguments, Trainer
    
    print("\n⚙️ Setting up training...")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=100,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=2,
        fp16=True,
        report_to=["tensorboard"],
        logging_dir=f"{OUTPUT_DIR}/logs",
        remove_unused_columns=False,
        dataloader_pin_memory=True,
        optim="paged_adamw_32bit",
        seed=42
    )
    
    print(f"✅ Training config ready")
    print(f"   Batch size: {training_args.per_device_train_batch_size} (effective: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps})")
    print(f"   Learning rate: {training_args.learning_rate}")
    print(f"   Epochs: {training_args.num_train_epochs}")
    
    print("\n🚀 Starting SFT training...")
    print(f"⏱️  ETA: ~{len(dataset) * 3 / 32 / 60:.1f} minutes\n")
    
    trainer = Trainer(
        model=model,
        train_dataset=dataset,
        args=training_args,
    )
    
    try:
        train_result = trainer.train()
        print("\n✅ SFT Training complete!")
        print(f"   Final loss: {train_result.training_loss:.4f}")
        
        # Save model
        model_dir = f"{OUTPUT_DIR}/finetuned_model"
        os.makedirs(model_dir, exist_ok=True)
        model.save_pretrained(model_dir)
        tokenizer.save_pretrained(model_dir)
        print(f"✅ Model saved to: {model_dir}")
        
        sft_training_loss = train_result.training_loss
        sft_trained = True
        
    except Exception as e:
        print(f"❌ Training error: {e}")
        sft_trained = False
else:
    sft_trained = False

---

# 🟠 PART 2: REINFORCEMENT LEARNING (RL)
*Train PPO policy on CyberMultiAgentEnv*

---

## ✅ RL Step 1: Load Cases & Initialize Agents

In [ ]:
if RUN_RL_TRAINING:
    print("\n" + "="*70)
    print("🟠 RL TRAINING: Environment Setup")
    print("="*70)
    
    sys.path.insert(0, '/content/MetaHackUI')
    
    # Load cases
    cases_path = "/content/MetaHackUI/dataset/cases.json"
    if os.path.exists(cases_path):
        with open(cases_path, 'r') as f:
            cases = json.load(f)
        print(f"✅ Loaded {len(cases)} incident cases")
    else:
        print(f"⚠️ {cases_path} not found - using empty cases")
        cases = []
    
    # Initialize agents
    print("\n🤖 Initializing agents...")
    try:
        from agents import CodeAgent, CriticAgent, FusionAgent, LogAgent, ReqAgent
        from evaluator import Evaluator
        from schemas import PolicyState
        
        agents_dict = {
            "log_agent": LogAgent(),
            "code_agent": CodeAgent(),
            "req_agent": ReqAgent(),
            "fusion_agent": FusionAgent(),
            "critic_agent": CriticAgent(),
        }
        
        evaluator = Evaluator()
        policy = PolicyState(
            log_sensitivity=0.5,
            code_sensitivity=0.5,
            req_sensitivity=0.5,
            fusion_temperature=0.7,
            confidence_threshold=0.5
        )
        
        print("✅ All agents initialized")
        rl_ready = True
        
    except Exception as e:
        print(f"⚠️ Agent initialization error (may be OK for this demo): {e}")
        rl_ready = False
else:
    rl_ready = False

## ✅ RL Step 2: Create Environment & Setup PPO

In [ ]:
if RUN_RL_TRAINING and rl_ready:
    from openenv import CyberMultiAgentEnv
    from stable_baselines3 import PPO
    from stable_baselines3.common.callbacks import BaseCallback
    import numpy as np
    
    print("\n🌍 Creating training environment...")
    
    # Create environment
    env = CyberMultiAgentEnv(
        cases=cases[:50] if cases else [],
        agents_dict=agents_dict,
        evaluator=evaluator,
        policy_state=policy
    )
    
    print("✅ Environment created")
    
    # Custom callback
    class MetricsCallback(BaseCallback):
        def __init__(self, log_dir):
            super().__init__()
            self.log_dir = log_dir
            self.episode_rewards = []
            self.episode_lengths = []
            self.current_episode_reward = 0
            self.current_episode_length = 0
            
        def _on_step(self) -> bool:
            self.current_episode_reward += self.locals["rewards"][0]
            self.current_episode_length += 1
            
            if self.locals["dones"][0]:
                self.episode_rewards.append(self.current_episode_reward)
                self.episode_lengths.append(self.current_episode_length)
                
                if len(self.episode_rewards) % 10 == 0:
                    mean_reward = np.mean(self.episode_rewards[-10:])
                    print(f"  Episode {len(self.episode_rewards)}: reward={self.current_episode_reward:.4f}, mean(last 10)={mean_reward:.4f}")
                
                self.current_episode_reward = 0
                self.current_episode_length = 0
            
            return True
    
    os.makedirs(RL_OUTPUT_DIR, exist_ok=True)
    callback = MetricsCallback(log_dir=RL_OUTPUT_DIR)
    
    print("⚙️ Creating PPO model...")
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        verbose=1,
        tensorboard_log=f"{RL_OUTPUT_DIR}/tb_logs",
    )
    print("✅ PPO model created")
    
    rl_model = model
    rl_callback = callback
else:
    rl_model = None
    rl_callback = None

## ✅ RL Step 3: Train PPO Model

In [ ]:
if RUN_RL_TRAINING and rl_model and rl_callback:
    print("\n🚀 Starting PPO training...")
    print("⏱️  ETA: ~5-10 minutes for 10000 timesteps\n")
    
    try:
        rl_model.learn(
            total_timesteps=10000,
            callback=rl_callback,
            log_interval=100,
            progress_bar=True
        )
        
        print("\n✅ RL Training complete!")
        print(f"   Total episodes: {len(rl_callback.episode_rewards)}")
        if rl_callback.episode_rewards:
            print(f"   Mean reward: {np.mean(rl_callback.episode_rewards):.4f}")
            print(f"   Max reward: {np.max(rl_callback.episode_rewards):.4f}")
        
        # Save model
        model_path = f"{RL_OUTPUT_DIR}/ppo_policy"
        rl_model.save(model_path)
        print(f"✅ Model saved: {model_path}.zip")
        
        rl_trained = True
        
    except Exception as e:
        print(f"❌ Training error: {e}")
        rl_trained = False
else:
    rl_trained = False

---

# 📊 PART 3: RESULTS & VISUALIZATION
*Generate graphs and summaries*

---

## ✅ Generate SFT Training Graphs

In [ ]:
if RUN_SFT_TRAINING and sft_trained:
    import matplotlib.pyplot as plt
    
    print("\n📊 Generating SFT training graphs...")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("SFT Training Metrics - Qwen2.5-3B LoRA Fine-tuning", fontsize=16, fontweight='bold')
    
    # Plot 1: Loss (placeholder - check TensorBoard for actual)
    ax = axes[0, 0]
    ax.text(0.5, 0.5, "Check TensorBoard for live loss curves", ha='center', va='center', fontsize=12)
    ax.set_title("Training Loss", fontweight='bold')
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    
    # Plot 2: Learning rate
    ax = axes[0, 1]
    steps = [i for i in range(1, 101)]
    lrs = [2e-4 * min(i/100, 1.0) for i in steps]
    ax.plot(steps, lrs, 'g-', linewidth=2)
    ax.set_title("Learning Rate Schedule", fontweight='bold')
    ax.set_xlabel("Step (relative)")
    ax.set_ylabel("Learning Rate")
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Config
    ax = axes[1, 0]
    ax.axis('off')
    config_text = f"""
SFT Training Configuration:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Model: Qwen2.5-3B-Instruct
LoRA Rank: 8
Batch Size: 1 (Accum: 8)
Learning Rate: 2e-4
Epochs: 3
Max Length: 512

System:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}
VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}GB if torch.cuda.is_available() else 'N/A'}
FP16: Enabled
Gradient Checkpointing: On
"""
    ax.text(0.1, 0.9, config_text, fontfamily='monospace', fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    # Plot 4: Results
    ax = axes[1, 1]
    ax.axis('off')
    results_text = f"""
SFT Training Results:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Dataset: train.jsonl
Final Loss: {sft_training_loss:.4f}
Epochs Completed: 3

Saved Artifacts:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Model: {OUTPUT_DIR}/finetuned_model
✅ Logs: {OUTPUT_DIR}/logs
✅ Stats: {OUTPUT_DIR}/training_stats.json
"""
    ax.text(0.1, 0.9, results_text, fontfamily='monospace', fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
    
    plt.tight_layout()
    graph_path = f"{OUTPUT_DIR}/training_metrics.png"
    plt.savefig(graph_path, dpi=150, bbox_inches='tight')
    print(f"✅ Graph saved: {graph_path}")
    plt.show()
    
    # Save metrics JSON
    sft_metrics = {
        "model": "Qwen/Qwen2.5-3B-Instruct",
        "training_epochs": 3,
        "final_loss": float(sft_training_loss),
        "checkpoint_path": f"{OUTPUT_DIR}/finetuned_model"
    }
    with open(f"{OUTPUT_DIR}/training_stats.json", 'w') as f:
        json.dump(sft_metrics, f, indent=2)
else:
    print("⏭️ Skipping SFT graphs")

## ✅ Generate RL Training Graphs

In [ ]:
if RUN_RL_TRAINING and rl_trained and rl_callback:
    from scipy.ndimage import uniform_filter1d
    
    print("\n📊 Generating RL training graphs...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("RL Training Metrics - PPO on CyberMultiAgentEnv", fontsize=16, fontweight='bold')
    
    # Plot 1: Episode Rewards
    ax = axes[0, 0]
    if rl_callback.episode_rewards:
        episodes = np.arange(len(rl_callback.episode_rewards))
        ax.plot(episodes, rl_callback.episode_rewards, 'b-', alpha=0.5, linewidth=1, label='Reward')
        
        if len(rl_callback.episode_rewards) > 10:
            moving_avg = uniform_filter1d(rl_callback.episode_rewards, size=10)
            ax.plot(episodes, moving_avg, 'r-', linewidth=2.5, label='Moving Avg (10)')
        
        ax.set_title("Episode Rewards", fontweight='bold', fontsize=12)
        ax.set_xlabel("Episode")
        ax.set_ylabel("Reward")
        ax.grid(True, alpha=0.3)
        ax.legend()
    
    # Plot 2: Episode Length Distribution
    ax = axes[0, 1]
    if rl_callback.episode_lengths:
        ax.hist(rl_callback.episode_lengths, bins=20, color='green', alpha=0.7, edgecolor='black')
        ax.set_title("Episode Length Distribution", fontweight='bold', fontsize=12)
        ax.set_xlabel("Steps per Episode")
        ax.set_ylabel("Frequency")
        ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Cumulative Reward
    ax = axes[1, 0]
    if rl_callback.episode_rewards:
        cumulative_reward = np.cumsum(rl_callback.episode_rewards)
        ax.plot(cumulative_reward, 'g-', linewidth=2)
        ax.fill_between(np.arange(len(cumulative_reward)), cumulative_reward, alpha=0.3, color='green')
        ax.set_title("Cumulative Reward Over Episodes", fontweight='bold', fontsize=12)
        ax.set_xlabel("Episode")
        ax.set_ylabel("Cumulative Reward")
        ax.grid(True, alpha=0.3)
    
    # Plot 4: Summary
    ax = axes[1, 1]
    ax.axis('off')
    
    summary_text = f"""
PPO Training Configuration:
━━━━━━━━━━━━━━━━━━━━━━━━━━━
Algorithm: PPO
Learning Rate: 3e-4
N Steps: 2048
Batch Size: 64
Epochs: 10
Gamma: 0.99
GAE Lambda: 0.95

Results:
━━━━━━━━━━━━━━━━━━━━━━━━━━━
Episodes: {len(rl_callback.episode_rewards)}
Mean Reward: {np.mean(rl_callback.episode_rewards):.4f if rl_callback.episode_rewards else 'N/A'}
Max Reward: {np.max(rl_callback.episode_rewards):.4f if rl_callback.episode_rewards else 'N/A'}
Avg Episode Length: {np.mean(rl_callback.episode_lengths):.1f if rl_callback.episode_lengths else 'N/A'}

Artifacts:
━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Model: {RL_OUTPUT_DIR}/ppo_policy.zip
✅ TensorBoard: {RL_OUTPUT_DIR}/tb_logs
"""
    
    ax.text(0.05, 0.95, summary_text, fontfamily='monospace', fontsize=9.5,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))
    
    plt.tight_layout()
    graph_path = f"{RL_OUTPUT_DIR}/training_graphs.png"
    plt.savefig(graph_path, dpi=150, bbox_inches='tight')
    print(f"✅ Graph saved: {graph_path}")
    plt.show()
    
    # Save metrics JSON
    rl_metrics = {
        "model_type": "PPO",
        "total_timesteps": 10000,
        "episodes_trained": len(rl_callback.episode_rewards),
        "mean_episode_reward": float(np.mean(rl_callback.episode_rewards)) if rl_callback.episode_rewards else 0,
        "max_episode_reward": float(np.max(rl_callback.episode_rewards)) if rl_callback.episode_rewards else 0,
    }
    with open(f"{RL_OUTPUT_DIR}/training_metrics.json", 'w') as f:
        json.dump(rl_metrics, f, indent=2)
else:
    print("⏭️ Skipping RL graphs")

## ✅ Final Summary

In [ ]:
print("\n" + "="*70)
print("🎯 TRAINING COMPLETE")
print("="*70)

summary = "\n✅ Completed:\n"
if sft_trained:
    summary += f"  ✅ SFT Fine-tuning: Qwen2.5-3B on {len(dataset) if dataset else '?'} examples\n"
    summary += f"     Output: {OUTPUT_DIR}/\n"
else:
    summary += "  ⏭️ SFT Training: Skipped\n"

if rl_trained:
    summary += f"  ✅ RL Training: PPO on {len(rl_callback.episode_rewards)} episodes\n"
    summary += f"     Output: {RL_OUTPUT_DIR}/\n"
else:
    summary += "  ⏭️ RL Training: Skipped\n"

summary += "\n📥 Download from Google Drive:\n"
summary += f"   • {OUTPUT_DIR}/\n"
summary += f"   • {RL_OUTPUT_DIR}/\n"

summary += "\n📊 Generated Files:\n"
if sft_trained:
    summary += "   • training_metrics.png (SFT graphs)\n"
    summary += "   • training_stats.json (SFT metrics)\n"
if rl_trained:
    summary += "   • training_graphs.png (RL graphs)\n"
    summary += "   • training_metrics.json (RL metrics)\n"

summary += "\n🚀 Next Steps:\n"
summary += "   1. Download models from Google Drive\n"
summary += "   2. Review graphs (PNG files)\n"
summary += "   3. Use models in your detection pipeline\n"

print(summary)
print("="*70)